In [1]:
import os
import yfinance as yf
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

TICKER = "RELIANCE.NS"

In [2]:
def fetch_stock_data(ticker, start="2022-01-01", end=None):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)

    if df is None or df.empty:
        raise ValueError(f"No data found for ticker: {ticker}")

    df = df.reset_index()
    return df

In [3]:
 def add_technical_features(df):
    df = df.copy()

    df["return_1d"] = df["Close"].pct_change()
    df["return_5d"] = df["Close"].pct_change(5)

    df["sma_5"] = df["Close"].rolling(5).mean()
    df["sma_10"] = df["Close"].rolling(10).mean()

    df["ema_10"] = df["Close"].ewm(span=10).mean()

    df["volatility_5"] = df["return_1d"].rolling(5).std()
    df["volume_change"] = df["Volume"].pct_change()

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    df["rsi_14"] = 100 - (100 / (1 + rs))

    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    df = df.dropna().reset_index(drop=True)

    return df

In [4]:
def prepare_dataset(ticker):
    df = fetch_stock_data(ticker)
    df = add_technical_features(df)
    return df

In [6]:
def add_technical_features(df):
    df = df.copy()

    df["return_1d"] = df["Close"].pct_change()
    df["return_5d"] = df["Close"].pct_change(5)

    df["sma_5"] = df["Close"].rolling(5).mean()
    df["sma_10"] = df["Close"].rolling(10).mean()

    df["ema_10"] = df["Close"].ewm(span=10).mean()

    df["volatility_5"] = df["return_1d"].rolling(5).std()
    df["volume_change"] = df["Volume"].pct_change()

    # RSI (main source of inf)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()

    rs = gain / loss
    df["rsi_14"] = 100 - (100 / (1 + rs))

    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    # ✅ FIX HERE
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df = df.dropna().reset_index(drop=True)

    return df

In [10]:
 # Select features
X = df[feature_cols]
y = df["target"]

# 🚨 CRITICAL CLEANING STEP
X = X.replace([np.inf, -np.inf], np.nan)

# Combine X and y to drop same rows
data = pd.concat([X, y], axis=1).dropna()

X = data[feature_cols]
y = data["target"]

KeyError: "None of [Index(['return_1d', 'return_5d', 'sma_5', 'sma_10', 'ema_10', 'volatility_5',\n       'volume_change', 'rsi_14'],\n      dtype='str')] are in the [columns]"

In [7]:
def backtest_strategy(df, model, feature_cols):
    df = df.copy()

    # Predict on full dataset
    df["prediction"] = model.predict(df[feature_cols])

    # Strategy: buy when prediction = 1
    df["strategy_return"] = df["prediction"].shift(1) * df["return_1d"]

    # Market return
    df["market_return"] = df["return_1d"]

    # Cumulative returns
    df["cum_strategy"] = (1 + df["strategy_return"]).cumprod()
    df["cum_market"] = (1 + df["market_return"]).cumprod()

    return df

In [8]:
df_bt = backtest_strategy(df, model, feature_cols)

AttributeError: 'RandomForestClassifier' object has no attribute 'estimators_'